# Valores que faltan: qué modelo usar y cuándo no calcular

Una analítica no trae siempre los mismos valores; lo que falta se rellena con la mediana de su sexo y grupo de edad. Dos preguntas:

1. **¿Desde cuántos Extras compensa su modelo** frente al de Routine (y `full` frente a `questionnaire`)?
2. **¿Con cuántos valores de Routine se deja de calcular?**

Método: se coge a gente **con todos los valores**, se esconden algunos y se mira cuánto se aleja el resultado del que daría conociéndolos todos. Cada persona se evalúa con un modelo entrenado sin ella (los 5 bloques de train); el test no se toca.

Criterios, fijados antes de ver los resultados:

| Pregunta | Criterio |
|---|---|
| 1 | El modelo con Extras compensa con *k* valores si sube al menos **0,002** el C-index misma edad frente al modelo sin Extras (el listón de siempre) **y** su edad se queda más cerca de la de los 8 Extras (menor mediana de la diferencia) |
| 2 | Se deja de calcular cuando a 1 de cada 10 personas la edad se le mueve más de *X* años. **X lo decide Rubén**; se enseña todo para 2, 3, 4 y 5 |

In [1]:
# Imports
import time
from typing import get_args

import numpy as np
import pandas as pd

from bioage.config.model_levels import BASELINE, EXTRAS, LEVEL_COLUMNS, ROUTINE, LabVariable
from bioage.config.paths import GOLD_TABLE
from bioage.modeling.s1_split import split_data
from bioage.modeling.s3_algorithms import build_pipeline, survival_target
from bioage.modeling.s4_biological_age import biological_age, fit_reference, risk_at_horizon
from bioage.modeling.s5_evaluation import c_index_by_group, same_age_c_index
from bioage.utils.features import age_groups

C:\Data\1_MisCosas\Evolve\TFM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
REPEATS = 3  # random choices of what is hidden
SEED = 42
LIMITS = [2, 3, 4, 5]  # years
ROUTINE_LAB = [column for column in get_args(LabVariable) if column in ROUTINE]

gold = pd.read_parquet(GOLD_TABLE)
train, test, folds = split_data(gold)
people = {level: train.dropna(subset=columns) for level, columns in LEVEL_COLUMNS.items()}
print(f"Train: {len(train)} people; with every value of each level:")
print({level: len(known) for level, known in people.items()})
print(f"Routine lab values: {len(ROUTINE_LAB)}; Extras: {len(EXTRAS)}")

Train: 38795 people; with every value of each level:
{'routine': 37361, 'extras': 33170, 'questionnaire': 23704, 'full': 22880}
Routine lab values: 15; Extras: 8


## 1. Modelos

Un modelo por nivel y bloque, con su curva de referencia. El de edad y sexo (nivel 0) sirve de suelo: es lo que se sabe sin analítica.

In [3]:
start = time.time()
fitted = {level: [] for level in ["baseline", *LEVEL_COLUMNS]}
for level, columns in {"baseline": BASELINE, **LEVEL_COLUMNS}.items():
    for fit_rows, validation_rows in folds:
        fit = train.iloc[fit_rows]
        pipeline = build_pipeline("cox", columns).fit(fit, survival_target(fit))
        reference = fit_reference(risk_at_horizon(pipeline, fit), fit)
        fitted[level].append((train.index[validation_rows], pipeline, reference))
    print(f"{level}: fitted ({(time.time() - start) / 60:.0f} min)", flush=True)

baseline: fitted (1 min)


routine: fitted (5 min)


extras: fitted (9 min)


questionnaire: fitted (12 min)


full: fitted (16 min)


In [4]:
def predict(level: str, data: pd.DataFrame, hidden: pd.DataFrame | None = None) -> pd.DataFrame:
    """Risk and biological age of each person, each from the models that did not see them."""
    if hidden is not None:
        data = data.assign(**{column: data[column].mask(hidden[column]) for column in hidden})
    parts = []
    for validation, pipeline, reference in fitted[level]:
        members = data.loc[data.index.isin(validation)]
        risk = risk_at_horizon(pipeline, members)
        age = biological_age(risk, members["sex"], reference)
        parts.append(pd.DataFrame({"risk": risk, "age": age}, index=members.index))
    return pd.concat(parts).loc[data.index]


def hide_random(data: pd.DataFrame, columns: list[str], count: int, seed: int) -> pd.DataFrame:
    """For each person, which of the columns are hidden: count of them, at random."""
    order = np.random.default_rng(seed).random((len(data), len(columns))).argsort(axis=1).argsort(axis=1)
    return pd.DataFrame(order < count, index=data.index, columns=columns)


def hide_all(data: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """The same columns hidden for everybody."""
    return pd.DataFrame(True, index=data.index, columns=columns)


def same_age(risk: pd.Series, data: pd.DataFrame) -> float:
    """C-index among people of the same age."""
    return same_age_c_index(c_index_by_group(risk, data, age_groups(data)))


def moved(age: pd.Series, target: pd.Series) -> dict[str, float]:
    """How far the age is from the one with every value."""
    change = age - target
    return {
        "median |change|": change.abs().median(),
        "p90 |change|": change.abs().quantile(0.9),
        "p95 |change|": change.abs().quantile(0.95),
        "mean change": change.mean(),
        **{f"over {years} years": (change.abs() > years).mean() for years in LIMITS},
        "over 3 years younger": (change < -3).mean(),
    }

## 2. ¿Desde cuántos Extras compensa su modelo?

`known` es cuántos de los 8 Extras se conocen, elegidos al azar para cada persona (media de tres sorteos). `no extras` es el modelo sin Extras. Las columnas de años dicen cuánto se aleja cada uno de la edad con los 8.

In [5]:
def extras_table(with_extras: str, without_extras: str) -> pd.DataFrame:
    data = people[with_extras]
    target = predict(with_extras, data)
    simple = predict(without_extras, data)
    rows = {"no extras": {"c_index_same_age": same_age(simple["risk"], data), **moved(simple["age"], target["age"])}}
    for known in range(len(EXTRAS) + 1):
        repeats = []
        for repeat in range(REPEATS if 0 < known < len(EXTRAS) else 1):
            result = predict(with_extras, data, hide_random(data, EXTRAS, len(EXTRAS) - known, SEED + repeat))
            repeats.append({"c_index_same_age": same_age(result["risk"], data), **moved(result["age"], target["age"])})
        rows[known] = pd.DataFrame(repeats).mean()
        print(f"{with_extras}: {known} known ({(time.time() - start) / 60:.0f} min)", flush=True)
    table = pd.DataFrame(rows).T
    table.insert(1, "gain", table["c_index_same_age"] - table.loc["no extras", "c_index_same_age"])
    return table


extras_tables = {"extras": extras_table("extras", "routine"), "full": extras_table("full", "questionnaire")}

extras: 0 known (16 min)


extras: 1 known (17 min)


extras: 2 known (17 min)


extras: 3 known (17 min)


extras: 4 known (18 min)


extras: 5 known (19 min)


... (8 outputs left out)

full: 5 known (23 min)


full: 6 known (24 min)


full: 7 known (25 min)


full: 8 known (25 min)


In [6]:
extras_tables["extras"].round(4)

,c_index_same_age,gain,median |change|,p90 |change|,p95 |change|,mean change,over 2 years,over 3 years,over 4 years,over 5 years,over 3 years younger
no extras,0.6855,0.0000,3.1410,8.0460,9.7908,-0.7029,0.6659,0.5197,0.3945,0.2914,0.3041
0,0.6642,-0.0213,3.4756,9.0720,11.3220,-0.7821,0.6983,0.5626,0.4387,0.3391,0.3127
1,0.6748,-0.0107,3.1741,8.3616,10.3993,-0.6780,0.6689,0.5236,0.3995,0.2981,0.2922
2,0.6835,-0.0020,2.8676,7.5891,9.4625,-0.5779,0.6344,0.4812,0.3545,0.2533,0.2684
3,0.6909,0.0054,2.5496,6.8085,8.5128,-0.4819,0.5943,0.4287,0.2996,0.2046,0.2413
4,0.6969,0.0114,2.2001,5.9247,7.4720,-0.3782,0.5384,0.3664,0.2391,0.1521,0.2073
5,0.7015,0.0160,1.8360,5.0065,6.3808,-0.2827,0.4638,0.2858,0.1700,0.1003,0.1642
6,0.7078,0.0223,1.3996,3.9547,5.1064,-0.1830,0.3506,0.1845,0.0973,0.0533,0.1100
7,0.7121,0.0266,0.8445,2.6916,3.5660,-0.0931,0.1818,0.0768,0.0360,0.0185,0.0486
8,0.7173,0.0318,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


In [7]:
extras_tables["full"].round(4)

,c_index_same_age,gain,median |change|,p90 |change|,p95 |change|,mean change,over 2 years,over 3 years,over 4 years,over 5 years,over 3 years younger
no extras,0.7172,0.0000,2.6746,6.8018,8.2503,0.0792,0.6101,0.4503,0.3163,0.2121,0.2156
0,0.7029,-0.0143,3.0394,7.8152,9.6943,-0.3759,0.6579,0.5051,0.3772,0.2720,0.2635
1,0.7072,-0.0101,2.7782,7.2264,8.9618,-0.3202,0.6253,0.4675,0.3384,0.2373,0.2445
2,0.7124,-0.0049,2.5318,6.5851,8.1590,-0.2824,0.5920,0.4258,0.2942,0.1970,0.2243
3,0.7165,-0.0007,2.2439,5.9000,7.3396,-0.2315,0.5462,0.3721,0.2458,0.1546,0.1982
4,0.7208,0.0035,1.9477,5.1887,6.4818,-0.1931,0.4890,0.3102,0.1881,0.1101,0.1684
5,0.7244,0.0071,1.6258,4.4130,5.6192,-0.1435,0.4101,0.2353,0.1295,0.0706,0.1311
6,0.7293,0.0120,1.2495,3.5412,4.5058,-0.0948,0.3011,0.1472,0.0710,0.0365,0.0856
7,0.7323,0.0151,0.7211,2.4104,3.1594,-0.0478,0.1524,0.0576,0.0248,0.0115,0.0361
8,0.7354,0.0181,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


### Qué Extras pesan más

Con **solo ese** conocido y con **solo ese** escondido. `gain` es el C-index misma edad frente al modelo sin Extras.

In [8]:
data = people["extras"]
target = predict("extras", data)
floor = same_age(predict("routine", data)["risk"], data)
rows = {}
for extra in EXTRAS:
    only = predict("extras", data, hide_all(data, [column for column in EXTRAS if column != extra]))
    lacking = predict("extras", data, hide_all(data, [extra]))
    rows[extra] = {
        "gain with only it": same_age(only["risk"], data) - floor,
        "gain lacking it": same_age(lacking["risk"], data) - floor,
        "lacking it: median |change|": (lacking["age"] - target["age"]).abs().median(),
        "lacking it: p90 |change|": (lacking["age"] - target["age"]).abs().quantile(0.9),
    }
    print(f"{extra} ({(time.time() - start) / 60:.0f} min)", flush=True)
pd.DataFrame(rows).T.sort_values("gain with only it", ascending=False).round(4)

ggt (26 min)


alkaline_phosphatase (26 min)


albumin (27 min)


vitamin_d (27 min)


chloride (28 min)


phosphorus (28 min)


ldh (29 min)


urine_acr (29 min)


,gain with only it,gain lacking it,lacking it: median |change|,lacking it: p90 |change|
urine_acr,0.0008,0.0211,0.8131,2.8393
ggt,-0.0006,0.0249,1.2218,3.4334
albumin,-0.0088,0.0271,1.5795,3.9331
vitamin_d,-0.0128,0.0301,1.2387,2.9423
ldh,-0.0143,0.0289,0.8470,2.2082
chloride,-0.0149,0.0276,1.0078,2.4704
alkaline_phosphatase,-0.0170,0.0322,0.4067,1.0744
phosphorus,-0.0206,0.0310,0.5709,1.5169


## 3. ¿Con cuántos valores de Routine se deja de calcular?

`missing` es cuántos de los 15 valores de laboratorio faltan, al azar. Edad, sexo, IMC y hábitos se conocen siempre. `c_index_same_age` se compara con el de `age and sex`, que es lo que se sabe sin analítica.

In [9]:
def missing_table(level: str) -> pd.DataFrame:
    data = people[level]
    target = predict(level, data)
    rows = {"age and sex": {"c_index_same_age": same_age(predict("baseline", data)["risk"], data)}}
    for missing in range(len(ROUTINE_LAB) + 1):
        repeats = []
        for repeat in range(REPEATS if 0 < missing < len(ROUTINE_LAB) else 1):
            result = predict(level, data, hide_random(data, ROUTINE_LAB, missing, SEED + repeat))
            repeats.append({"c_index_same_age": same_age(result["risk"], data), **moved(result["age"], target["age"])})
        rows[missing] = pd.DataFrame(repeats).mean()
        print(f"{level}: {missing} missing ({(time.time() - start) / 60:.0f} min)", flush=True)
    return pd.DataFrame(rows).T


missing_tables = {level: missing_table(level) for level in ["routine", "questionnaire"]}

routine: 0 missing (30 min)


routine: 1 missing (31 min)


routine: 2 missing (32 min)


routine: 3 missing (33 min)


routine: 4 missing (34 min)


routine: 5 missing (34 min)


... (22 outputs left out)

questionnaire: 12 missing (48 min)


questionnaire: 13 missing (49 min)


questionnaire: 14 missing (49 min)


questionnaire: 15 missing (50 min)


In [10]:
missing_tables["routine"].round(3)

,c_index_same_age,mean change,median |change|,over 2 years,over 3 years,over 3 years younger,over 4 years,over 5 years,p90 |change|,p95 |change|
age and sex,0.596,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,0.684,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
1,0.677,-0.039,0.853,0.200,0.104,0.056,0.058,0.034,3.058,4.277
2,0.670,-0.083,1.425,0.363,0.211,0.111,0.125,0.077,4.457,5.975
3,0.662,-0.138,1.848,0.469,0.299,0.157,0.190,0.122,5.453,7.180
4,0.657,-0.176,2.175,0.534,0.365,0.191,0.245,0.164,6.238,8.121
5,0.651,-0.227,2.453,0.579,0.414,0.218,0.291,0.201,6.883,8.872
6,0.645,-0.259,2.692,0.614,0.455,0.238,0.329,0.234,7.411,9.471
7,0.638,-0.308,2.906,0.638,0.487,0.256,0.360,0.262,7.892,10.037
8,0.633,-0.345,3.064,0.658,0.508,0.267,0.384,0.284,8.260,10.444


In [11]:
missing_tables["questionnaire"].round(3)

,c_index_same_age,mean change,median |change|,over 2 years,over 3 years,over 3 years younger,over 4 years,over 5 years,p90 |change|,p95 |change|
age and sex,0.597,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,0.721,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
1,0.716,-0.052,0.658,0.146,0.067,0.038,0.034,0.019,2.476,3.405
2,0.712,-0.096,1.141,0.281,0.146,0.081,0.079,0.045,3.613,4.782
3,0.707,-0.148,1.499,0.380,0.216,0.118,0.124,0.074,4.403,5.814
4,0.703,-0.179,1.778,0.452,0.279,0.151,0.170,0.104,5.070,6.663
5,0.700,-0.217,2.012,0.503,0.327,0.176,0.209,0.132,5.622,7.255
6,0.696,-0.252,2.208,0.540,0.368,0.198,0.244,0.158,6.080,7.764
7,0.695,-0.287,2.368,0.567,0.400,0.215,0.271,0.182,6.469,8.235
8,0.692,-0.317,2.511,0.588,0.422,0.225,0.294,0.201,6.797,8.641


### Cuántos pueden faltar, según X

El mayor número de valores que pueden faltar sin que a 1 de cada 10 personas se le mueva la edad más de X años.

In [12]:
pd.DataFrame({
    level: {f"X = {years}": table.index[1:][table["p90 |change|"].iloc[1:] <= years].max() for years in LIMITS}
    for level, table in missing_tables.items()
})

,routine,questionnaire
X = 2,0,0
X = 3,0,1
X = 4,1,2
X = 5,2,3


### Qué valores pesan más

Solo falta ese valor.

In [13]:
data = people["routine"]
target = predict("routine", data)
rows = {}
for column in ROUTINE_LAB:
    result = predict("routine", data, hide_all(data, [column]))
    rows[column] = {"c_index_same_age": same_age(result["risk"], data), **moved(result["age"], target["age"])}
    print(f"{column} ({(time.time() - start) / 60:.0f} min)", flush=True)
pd.DataFrame(rows).T.sort_values("p90 |change|", ascending=False).round(3)

hemoglobin (50 min)


erythrocytes (50 min)


mchc (51 min)


neutrophils (51 min)


lymphocytes (51 min)


monocytes (51 min)


... (6 outputs left out)

alt (53 min)


sodium (54 min)


potassium (54 min)


,c_index_same_age,median |change|,p90 |change|,p95 |change|,mean change,over 2 years,over 3 years,over 4 years,over 5 years,over 3 years younger
alt,0.662,2.288,6.474,8.694,0.630,0.555,0.386,0.260,0.173,0.163
ast,0.660,2.015,5.797,8.151,-0.650,0.503,0.326,0.209,0.135,0.192
erythrocytes,0.670,1.694,4.337,5.321,0.034,0.427,0.239,0.126,0.063,0.116
neutrophils,0.674,1.518,3.896,4.717,0.017,0.380,0.196,0.092,0.039,0.097
mchc,0.676,1.356,3.604,4.472,-0.023,0.333,0.160,0.072,0.033,0.084
glucose,0.673,0.760,2.761,4.474,-0.404,0.156,0.090,0.059,0.041,0.078
urea,0.678,0.848,2.189,2.660,-0.079,0.126,0.033,0.010,0.003,0.023
creatinine,0.676,0.832,2.124,2.666,-0.125,0.119,0.034,0.014,0.009,0.028
lymphocytes,0.681,0.831,2.072,2.493,0.061,0.112,0.021,0.004,0.001,0.006
uric_acid,0.683,0.788,1.992,2.461,-0.111,0.099,0.023,0.007,0.001,0.021


### Faltan bloques enteros

Los valores no faltan al azar: lo normal es que una analítica traiga el bloque entero o no lo traiga.

In [14]:
BLOCKS = {
    "blood count": ["hemoglobin", "erythrocytes", "mchc", "neutrophils", "lymphocytes", "monocytes"],
    "white cells": ["neutrophils", "lymphocytes", "monocytes"],
    "kidney": ["creatinine", "urea", "uric_acid"],
    "liver": ["ast", "alt"],
    "sodium and potassium": ["sodium", "potassium"],
    "glucose and hdl": ["glucose", "hdl"],
    "biochemistry": [column for column in ROUTINE_LAB if column not in ["hemoglobin", "erythrocytes", "mchc", "neutrophils", "lymphocytes", "monocytes"]],
}
rows = {}
for block, columns in BLOCKS.items():
    result = predict("routine", data, hide_all(data, columns))
    rows[block] = {"values": len(columns), "c_index_same_age": same_age(result["risk"], data), **moved(result["age"], target["age"])}
pd.DataFrame(rows).T.sort_values("p90 |change|", ascending=False).round(3)

,values,c_index_same_age,median |change|,p90 |change|,p95 |change|,mean change,over 2 years,over 3 years,over 4 years,over 5 years,over 3 years younger
biochemistry,9.0,0.644,2.624,6.897,8.678,-0.640,0.608,0.442,0.312,0.213,0.259
blood count,6.0,0.661,2.280,5.775,7.018,0.069,0.552,0.381,0.247,0.150,0.184
white cells,3.0,0.669,1.849,4.764,5.815,-0.002,0.466,0.280,0.159,0.086,0.139
liver,2.0,0.671,1.709,4.540,5.676,0.055,0.433,0.252,0.139,0.076,0.120
kidney,3.0,0.675,1.416,3.604,4.412,-0.314,0.345,0.165,0.072,0.030,0.108
glucose and hdl,2.0,0.672,0.928,3.064,4.689,-0.321,0.199,0.103,0.065,0.044,0.083
sodium and potassium,2.0,0.680,0.792,2.029,2.478,-0.064,0.104,0.022,0.004,0.001,0.015


### ¿A quién se le mueve más?

La mitad de los valores escondida, por edad y por resultado: quien sale peor es quien más se juega.

In [15]:
result = predict("routine", data, hide_random(data, ROUTINE_LAB, len(ROUTINE_LAB) // 2, SEED))
change = result["age"] - target["age"]
gap = pd.cut(target["age"] - data["age"], [-np.inf, -5, 0, 5, 10, np.inf], labels=["5+ younger", "0-5 younger", "0-5 older", "5-10 older", "10+ older"])


def by(groups: pd.Series) -> pd.DataFrame:
    return pd.DataFrame({name: {"people": len(part), **moved(result["age"][part.index], target["age"][part.index])} for name, part in change.groupby(groups, observed=True)}).T


pd.concat({"age": by(age_groups(data)), "biological minus real age": by(gap)}).round(2)

people  median |change|  p90 |change|  \
age                       18-29         8685.0             3.21          8.50   
                          30-39         6178.0             3.02          7.89   
                          40-49         6459.0             2.91          7.78   
                          50-59         5783.0             2.80          7.73   
                          60-69         6195.0             2.66          7.43   
                          70-79         4061.0             2.59          7.13   
biological minus real age 5+ younger    6264.0             3.65          8.75   
                          0-5 younger  12248.0             2.38          6.38   
                          0-5 older    11429.0             2.48          6.49   
                          5-10 older    5132.0             3.73          9.09   
                          10+ older     2288.0             5.69         13.57   

                                       p95 |change|  mean change  \
age                       18-29               10.60         0.07   
                          30-39               10.17        -0.23   
                          40-49                9.81        -0.44   
                          50-59                9.84        -0.54   
                          60-69                9.40        -0.43   
                          70-79                9.11        -0.47   
biological minus real age 5+ younger          10.61         3.31   
                          0-5 younger          7.97         0.93   
                          0-5 older            8.21        -1.15   
                          5-10 older          11.23        -3.34   
                          10+ older           17.14        -5.76   

                                       over 2 years  over 3 years  \
age                       18-29                0.67          0.53   
                          30-39                0.65          0.50   
                          40-49                0.63          0.48   
                          50-59                0.62          0.47   
                          60-69                0.61          0.45   
                          70-79                0.59          0.44   
biological minus real age 5+ younger           0.72          0.58   
                          0-5 younger          0.57          0.40   
                          0-5 older            0.58          0.42   
                          5-10 older           0.72          0.59   
                          10+ older            0.83          0.75   

                                       over 4 years  over 5 years  \
age                       18-29                0.40          0.30   
                          30-39                0.38          0.28   
                          40-49                0.35          0.26   
                          50-59                0.34          0.25   
                          60-69                0.33          0.24   
                          70-79                0.32          0.22   
biological minus real age 5+ younger           0.46          0.35   
                          0-5 younger          0.28          0.18   
                          0-5 older            0.29          0.19   
                          5-10 older           0.47          0.36   
                          10+ older            0.65          0.56   

                                       over 3 years younger  
age                       18-29                        0.26  
                          30-39                        0.26  
                          40-49                        0.26  
                          50-59                        0.25  
                          60-69                        0.25  
                          70-79                        0.25  
biological minus real age 5+ younger                   0.06  
                          0-5 younger                  0.13  
                          0-5 olde

## 4. Conclusión

**Extras.** Con el criterio fijado antes (subir al menos 0,002 el C-index misma edad y quedarse más cerca de la edad con los 8):

| Modelo | Compensa desde | Con ese número |
|---|---|---|
| `extras` frente a `routine` | **3 Extras** | +0,0054 de C-index; mediana del cambio 2,55 años frente a 3,14 |
| `full` frente a `questionnaire` | **4 Extras** | +0,0035; 1,95 años frente a 2,67 |

Con 2 (o 3 con cuestionario) el modelo con Extras ordena peor que el que no los usa: rellenar tantos huecos lo estropea. Los que más pesan son la albúmina/creatinina en orina y la GGT.

**Mínimo de datos.** Rubén elige un criterio laxo, X = 7 años: pueden faltar **hasta 5 valores de Routine sin cuestionario y hasta 8 con él**. Con X = 3 no podría faltar casi ninguno.

**Lo que más mueve la edad:** que falte solo la AST o solo la ALT (hasta 6,5 años en 9 de cada 10 personas), porque en el modelo se compensan entre sí; si faltan las dos, se mueve menos (4,5). Y a quien más se le mueve es a quien sale muy lejos de su edad: con 10 o más años de más, el doble que al resto.

**En la app:** `MAX_MISSING`, `MIN_EXTRAS`, `MISSING_MARGIN` y `EXTRAS_MARGIN`, en `config/app.py`, salen de las tablas de las secciones 2 y 3 (mediana del cambio para el margen ±).